
# PhishU — Master Notebook (EDA • PCA • Tabular Models • Semantic Baselines)

This notebook orchestrates all analyses and models without changing individual scripts.

**Sections**  
1. Environment & Logging  
2. Data loading (single source of truth)  
3. EDA (src/data_analysis/EDA.py)  
4. PCA / ACP (src/data_analysis/ACP.py)  
5. Tabular models: Logistic Regression, Random Forest, XGBoost (src/models/*.py)  
6. Semantic baselines: TF-IDF+LinearSVC, FastText+LogReg (src/semantic_models/...)  
7. Summary comparison table


## 1) Environment & Logging

In [ ]:
from src.pipeline.logger import init_logging, get_logger, set_seed
init_logging("INFO")  # set "DEBUG" for more verbose logs
log = get_logger("Notebook", "INFO")
set_seed(42)
log.info("Notebook started.")


## 2) Data loading

In [ ]:

# Load from UCI via our shared DataLoader
from src.utils.data_loader import DataLoader

dl = DataLoader(dataset_id=967)  # PhiUSIIL Phishing URL dataset
X_all, y_series, meta = dl.get_xy_as_dataframes()

# Build a single DataFrame
df = X_all.copy()
df["label"] = y_series.astype(int).values  # ensure numeric labels

log.info(f"Loaded from UCI: {df.shape[0]} rows, {df.shape[1]} cols "
         f"(features={X_all.shape[1]})")
df.head(3)

## 3) EDA

In [ ]:

import pandas as pd
from src.data_analysis.EDA import run_eda

eda_out = run_eda(
    df=df,
    label_col="label",
    save_dir="outputs/eda",
    save_fig=True,
    show_fig=False
)

display(pd.Series(eda_out["shape"], index=["rows","cols"]).to_frame("shape"))
display(eda_out["dtypes_counts"].to_frame("count").T)
display(eda_out["label_distribution_pct"].to_frame("pct"))
if eda_out["preview_describe"].shape[0] > 0:
    display(eda_out["preview_describe"])


## 4) PCA / ACP

In [ ]:

from src.data_analysis.ACP import run_pca

pca_out = run_pca(
    df=df,
    label_col="label",
    n_components=0.95,
    save_dir="outputs/pca",
    save_fig=True,
    show_fig=False
)

log.info(f"PCA retained components: {pca_out['n_components_']} "
         f"(cumulative variance={pca_out['explained_variance_ratio'].sum():.3f})")
display(pca_out["components_df"].head(5))
display(pd.DataFrame({
    "PC1_top": pca_out["pc1_top_loadings"],
    "PC2_top": pca_out["pc2_top_loadings"]
}))


## 5) Tabular models

In [ ]:

from src.models.regressionlogistique import run_logistic_regression
from src.models.randomforest import run_random_forest
from src.models.XGboost import run_xgboost
from src.models.NN import run_neural_network

tabular_results = {}
common_features = (
    'URLLength','DomainLength','NoOfSubDomain','IsDomainIP',
    'NoOfLettersInURL','NoOfDegitsInURL','NoOfEqualsInURL',
    'NoOfQMarkInURL','NoOfAmpersandInURL','NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL','TLDLength'
)

log.info("Running Logistic Regression (tabular)...")
logreg_out = run_logistic_regression(
    df=df,
    label_col="label",
    features=common_features,
    sample_n=10000,
    save_dir="outputs/logreg",
    save_fig=True,
    show_fig=False
)
tabular_results["LogisticRegression"] = logreg_out["metrics"]
display(pd.Series(logreg_out["metrics"], name="LogisticRegression"))

log.info("Running Random Forest (tabular)...")
rf_out = run_random_forest(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/random_forest",
    save_fig=True,
    show_fig=False
)
tabular_results["RandomForest"] = rf_out["metrics"]
display(pd.Series(rf_out["metrics"], name="RandomForest"))

log.info("Running XGBoost (tabular)...")
xgb_out = run_xgboost(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/xgboost",
    save_fig=True,
    show_fig=False
)
tabular_results["XGBoost"] = xgb_out["metrics"]
display(pd.Series(xgb_out["metrics"], name="XGBoost"))

log.info("Running Neural Network (tabular)...")
nn_out = run_neural_network(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    early_stop=True,
    save_dir="outputs/neural_network",
    save_fig=True,
    show_fig=False
)
tabular_results["NeuralNetwork"] = nn_out["metrics"]
display(pd.Series(nn_out["metrics"], name="NeuralNetwork"))


## 6) Semantic baselines (TF-IDF+LinearSVC, FastText+LogReg)

In [ ]:
from src.semantic_models.phishing_url_semantic_baselines import (
    train_and_compare_semantic_baselines, PhiUSIILSpec
)

# Pass the same df already loaded earlier in the notebook:
sem_out = train_and_compare_semantic_baselines(
    spec=PhiUSIILSpec(dataset_id=967),
    df=df,
    test_size=0.2,
    random_state=42,
    run_char_tfidf=True,
    run_fasttext=True,
    fasttext_fast_mode=True
)

# Build a compact metrics table
def _extract_metrics(res_dict):
    keep = ("accuracy","precision","recall","f1","roc_auc","pr_auc")
    table = {}
    for k, v in res_dict.items():
        if k in ("char_tfidf_svc", "fasttext_logreg"):
            table[k] = {m: float(v.get(m, float("nan"))) for m in keep}
    return table

sem_metrics = _extract_metrics(sem_out)
import pandas as pd
pd.DataFrame(sem_metrics).T


In [ ]:
tfidf = sem_out["char_tfidf_svc"]["vectorizer"]
svc   = sem_out["char_tfidf_svc"]["clf"]
names = np.array(tfidf.get_feature_names_out())
coefs = svc.coef_[0]

idx_pos = np.argsort(coefs)[-10:][::-1]
idx_neg = np.argsort(coefs)[:10]

print("Top phishing n-grams:", ", ".join(names[idx_pos]))
print("Top legitimate n-grams:", ", ".join(names[idx_neg]))

Top phishing n-grams: .gal, .mil, .gov, .ac., .gr, imate, .com, .it, .hr, .jp
Top legitimate n-grams: .cf, om/, com/, .com/, .ml, .ga, .gq, .top, .fr/, fr/


In [14]:
from collections import Counter
from src.semantic_models.phishing_url_semantic_baselines import (
    URLWordTokenTransformer, load_urls_and_labels, PhiUSIILSpec
)
from sklearn.model_selection import train_test_split

# Recover same split and tokenization
urls_all, y_all, _ = load_urls_and_labels(PhiUSIILSpec(dataset_id=967), df=df)
Xtr, Xte, ytr, yte = train_test_split(urls_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

tok_tf = sem_out["fasttext_logreg"]["token_transformer"]
ft_model = sem_out["fasttext_logreg"]["embedder"].model

tokens_test = tok_tf.transform(Xte)
y_test_np = yte.to_numpy()

# --- Token frequencies per class ---
phish_counter, legit_counter = Counter(), Counter()
for toks, lab in zip(tokens_test, y_test_np):
    toks = [t for t in toks if t in ft_model.wv and t.isalnum() and len(t) > 2]
    if lab == 1:
        phish_counter.update(toks)
    else:
        legit_counter.update(toks)

# Remove tokens that appear in both (uninformative overlap)
shared_tokens = set(phish_counter.keys()) & set(legit_counter.keys())
for token in shared_tokens:
    del phish_counter[token]
    del legit_counter[token]

# Now get top distinctive tokens
top_phish = [w for w, _ in phish_counter.most_common(20)]
top_legit = [w for w, _ in legit_counter.most_common(20)]

print(f"FastText vocab size: {len(ft_model.wv)}")
print("\nDistinct phishing-tilted tokens:", ", ".join(top_phish))
print("Distinct legitimate-tilted tokens:", ", ".join(top_legit))

# --- Nearest neighbors for a few representative tokens ---
def nn(token, k=5):
    try:
        return ", ".join([f"{w}({s:.2f})" for w, s in ft_model.wv.most_similar(token, topn=k)])
    except KeyError:
        return "(OOV)"

probe = top_phish[:3] + top_legit[:3]
print("\nNearest neighbors:")
for t in probe:
    print(f"  {t:>15s} -> {nn(t)}")


Tokenizing URLs (word tokens):   0%|          | 0/47159 [00:00<?, ?it/s]

FastText vocab size: 8574

Distinct phishing-tilted tokens: county, jewelry, schools, kids, jewelers, chicago, awards, village, tree, aviation, wine, council, aero, airport, university, utah, tips, york, independent, saint
Distinct legitimate-tilted tokens: ipfs, repl, pfs, att, godaddy, akc, https, blogspot, dweb, glitch, xsph, fleek, assets, webmail, signin, webwave, account, facebook, wixsite, weebly

Nearest neighbors:
           county -> council(0.91), count(0.85), courts(0.85), counter(0.81), countryside(0.81)
          jewelry -> jewels(0.95), jewelers(0.89), jewellery(0.89), jewellers(0.89), jeff(0.86)
          schools -> school(0.96), scholars(0.90), preschool(0.90), george(0.87), civic(0.86)
             ipfs -> pfs(0.86), mfs(0.83), afk(0.82), infura(0.79), ffs(0.78)
             repl -> reply(0.85), pl(0.80), rep(0.78), re(0.77), asd156(0.76)
              pfs -> flare(0.87), ipfs(0.86), mfs(0.85), infura(0.85), fur(0.85)


## 7) Summary comparison table

In [ ]:
import numpy as np
import pandas as pd

metrics = ["accuracy","precision","recall","f1","roc_auc","pr_auc"]

# --- Tabular models (PR-AUC may be missing → stays NaN) ---
tab_df = pd.DataFrame(tabular_results).T.reindex(columns=metrics)

# --- Semantic models (pull same metric set) ---
sem_df = pd.DataFrame({
    k: {m: sem_out[k].get(m, np.nan) for m in metrics}
    for k in sem_out
    if k in ("char_tfidf_svc","fasttext_logreg")
}).T

# --- Combine (all numeric; NaN where not available) ---
combined_df = pd.concat([tab_df, sem_df], axis=0)

# --- Nice formatting: 4 decimals for numbers, "—" for NaN ---
display(
    combined_df
      .style
      .format({col: "{:.4f}" for col in combined_df.columns}, na_rep="—")
      .set_caption("Overall model comparison (tabular + semantic)")
)
